# Practice: MNIST Image Classification with Three Models

Train an MLP, a simple CNN and a pretrained ResNet18 on the same set of handwritten digit images. Compare all three models on the same 48 test images.

Run this notebook from top to bottom in a fresh Colab runtime. For local Jupyter, make sure `sample_data/` exists in the working directory, `curl` and `unzip` are available, and the required packages are installed (`python -m pip install numpy torch torchvision scikit-learn`). The first ResNet18 run downloads about 45 MB of pretrained weights.

[Open in Colab](https://colab.research.google.com/github/wlos/ai_with_python_course/blob/main/16-Deep-Learning-and-Image-Processing/practice/Practice%20MNIST%20Image%20Classification.ipynb)


## Download and prepare the shared images

Download and extract the same PNG files used in Sections 6 and 7. Use the seed from those sections to reserve 192 images for training and 48 for testing.


In [ ]:
!curl -fLsS -o sample_data/digit_images.zip https://raw.githubusercontent.com/wlos/AI_SS/main/13-Computer-Vision/sample_data/digit_images.zip
!unzip -q -n sample_data/digit_images.zip


In [ ]:
from torchvision import datasets, transforms

DIGIT_ROOT = "sample_data/digit_images"

digit_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Use one intensity channel.
    transforms.Resize((28, 28)),  # Give every image the same size.
    transforms.ToTensor(),  # Scale 8-bit pixels to [0, 1] in (C, H, W) order.
])
digit_dataset = datasets.ImageFolder(DIGIT_ROOT, transform=digit_transform)
sample_image, sample_label = digit_dataset[0]
print(f"Images: {len(digit_dataset)} | Classes: {digit_dataset.class_to_idx}")
print(f"One image: {tuple(sample_image.shape)} | Label: {sample_label}")


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split

DIGIT_SEED = 42

# Randomly assign each image to exactly one subset using a fixed seed.
train_size = int(0.8 * len(digit_dataset))
test_size = len(digit_dataset) - train_size
torch.manual_seed(DIGIT_SEED)
digit_train, digit_test = random_split(
    digit_dataset,
    [train_size, test_size],
)
digit_train_loader = DataLoader(
    digit_train, batch_size=16, shuffle=True,
)
digit_test_loader = DataLoader(digit_test, batch_size=16, shuffle=False)

batch_images, batch_labels = next(iter(digit_test_loader))
print(f"Training images: {len(digit_train)} | Test images: {len(digit_test)}")
print(f"Image batch: {tuple(batch_images.shape)} | Label batch: {tuple(batch_labels.shape)}")
print(f"Batch labels: {batch_labels.tolist()}")


## 1. Train and test an MLP

Flatten each 28 × 28 image to 784 features. Train the MLP on `digit_train` and record its accuracy on `digit_test`.


In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier

def flatten_digit_images(dataset):
    """Turn each greyscale image into a row of pixel values."""
    pixels, labels = [], []
    for image, label in dataset:
        pixels.append(image.flatten().numpy())
        labels.append(label)
    return np.stack(pixels), labels

mlp_train_pixels, mlp_train_labels = flatten_digit_images(digit_train)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64,), max_iter=300, random_state=DIGIT_SEED,
)
mlp_model.fit(mlp_train_pixels, mlp_train_labels)
print(f"Training input shape: {mlp_train_pixels.shape}")


In [ ]:
from sklearn.metrics import accuracy_score

mlp_test_pixels, mlp_test_labels = flatten_digit_images(digit_test)
mlp_test_predictions = mlp_model.predict(mlp_test_pixels)
mlp_test_accuracy = accuracy_score(mlp_test_labels, mlp_test_predictions)
print(f"MLP test accuracy: {mlp_test_accuracy:.1%} on {len(mlp_test_labels)} images")


## 2. Train and test the simple CNN

Use the two convolution–ReLU–pooling blocks from Section 6. Train for the same number of epochs as the lesson and calculate test accuracy using the greyscale image tensors.


In [ ]:
from torch import nn

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

class DigitCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Reduce 28 × 28 to 14 × 14.
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Reduce 14 × 14 to 7 × 7.
        )
        self.flatten = nn.Flatten()  # Keep the batch dimension separate.
        self.head = nn.Linear(16 * 7 * 7, num_classes)

    def forward(self, images):
        features = self.features(images)
        return self.head(self.flatten(features))

torch.manual_seed(DIGIT_SEED)  # Reproduce the model's initial weights.
digit_class_count = len(digit_dataset.classes)
digit_cnn = DigitCNN(digit_class_count).to(DEVICE)


In [ ]:
DIGIT_EPOCHS = 10
DIGIT_LEARNING_RATE = 0.001
digit_loss = nn.CrossEntropyLoss()

def train_digit_epoch(model, loader, optimiser):
    '''Train for one epoch and return mean loss and accuracy.'''
    total_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimiser.zero_grad()  # Clear gradients left by the previous batch.
        logits = model(images)
        loss = digit_loss(logits, labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        loss.backward()  # Calculate gradients from the loss.
        optimiser.step()  # Apply an update to the optimiser's parameters.
        total_loss += loss.item() * len(labels)  # Weight each batch by its size.
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

digit_optimiser = torch.optim.Adam(digit_cnn.parameters(), lr=DIGIT_LEARNING_RATE)
digit_cnn.train()
cnn_training_losses = []
cnn_training_accuracies = []
for epoch in range(DIGIT_EPOCHS):
    mean_loss, train_accuracy = train_digit_epoch(digit_cnn, digit_train_loader, digit_optimiser)
    cnn_training_losses.append(mean_loss)
    cnn_training_accuracies.append(train_accuracy)
    print(f"Simple CNN epoch {epoch + 1}: loss={mean_loss:.3f} | accuracy={train_accuracy:.1%}")


In [ ]:
def predict_digits(model, loader):
    """Return true and predicted class indices in test-loader order."""
    model.eval()
    true_labels = []
    predictions = []
    with torch.no_grad():  # Inference does not need gradients.
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            true_labels.extend(labels.tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return true_labels, predictions

cnn_test_labels, cnn_test_predictions = predict_digits(digit_cnn, digit_test_loader)
print(f"Test predictions: {len(cnn_test_predictions)}")


In [ ]:
from sklearn.metrics import accuracy_score

simple_digit_accuracy = accuracy_score(cnn_test_labels, cnn_test_predictions)
print(f"Simple CNN test accuracy: {simple_digit_accuracy:.1%}")


## 3. Train and test the pretrained ResNet18

Apply the ResNet18 preprocessing to the same image files. Recreate the split with the same seed, freeze the backbone and train the replacement head as in Section 7. Then use the held-out images to measure accuracy.


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models

DIGIT_ROOT = "sample_data/digit_images"
DIGIT_SEED = 42

resnet_weights = models.ResNet18_Weights.DEFAULT
resnet_digit_dataset = datasets.ImageFolder(
    DIGIT_ROOT, transform=resnet_weights.transforms()
)
# Recreate the same split for the ResNet18 input transform.
train_size = int(0.8 * len(resnet_digit_dataset))
test_size = len(resnet_digit_dataset) - train_size
torch.manual_seed(DIGIT_SEED)
resnet_train, resnet_test = random_split(
    resnet_digit_dataset,
    [train_size, test_size],
)
resnet_train_loader = DataLoader(
    resnet_train,
    batch_size=16,
    shuffle=True,
)
resnet_test_loader = DataLoader(
    resnet_test,
    batch_size=16,
    shuffle=False,
)
resnet_image, _ = resnet_digit_dataset[0]
print(f"One ResNet18 input: {tuple(resnet_image.shape)}")


In [ ]:
from torch import nn

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
RESNET_LEARNING_RATE = 0.001

torch.manual_seed(DIGIT_SEED)
digit_resnet = models.resnet18(weights=resnet_weights)
digit_resnet.requires_grad_(False)  # Freeze all original pretrained parameters.
digit_resnet.fc = nn.Linear(
    digit_resnet.fc.in_features, len(resnet_digit_dataset.classes)
)
digit_resnet = digit_resnet.to(DEVICE)
digit_resnet.eval()  # Preserve the backbone's BatchNorm running statistics.
digit_resnet.fc.train()  # Set training mode only for the replacement head.
resnet_optimiser = torch.optim.Adam(digit_resnet.fc.parameters(), lr=RESNET_LEARNING_RATE)
print(f"Trainable head parameters: {sum(p.numel() for p in digit_resnet.fc.parameters()):,}")


In [ ]:
RESNET_EPOCHS = 10
resnet_loss = nn.CrossEntropyLoss()

digit_resnet.eval()  # Keep the frozen backbone in evaluation mode.
digit_resnet.fc.train()
resnet_training_losses = []
resnet_training_accuracies = []
for epoch in range(RESNET_EPOCHS):
    total_loss = 0.0
    correct = 0
    for images, labels in resnet_train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        resnet_optimiser.zero_grad()  # Clear gradients from the previous batch.
        logits = digit_resnet(images)
        loss = resnet_loss(logits, labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        loss.backward()
        resnet_optimiser.step()  # Update the replacement head.
        total_loss += loss.item() * len(labels)
    mean_loss = total_loss / len(resnet_train_loader.dataset)
    train_accuracy = correct / len(resnet_train_loader.dataset)
    resnet_training_losses.append(mean_loss)
    resnet_training_accuracies.append(train_accuracy)
    print(f"ResNet18 head epoch {epoch + 1}: loss={mean_loss:.3f} | accuracy={train_accuracy:.1%}")


In [ ]:
digit_resnet.eval()
resnet_test_labels = []
resnet_test_predictions = []
with torch.no_grad():  # Collect predictions without tracking gradients.
    for images, labels in resnet_test_loader:
        logits = digit_resnet(images.to(DEVICE))
        resnet_test_labels.extend(labels.tolist())
        resnet_test_predictions.extend(logits.argmax(dim=1).cpu().tolist())

print(f"Test predictions: {len(resnet_test_predictions)}")


In [ ]:
from sklearn.metrics import accuracy_score

resnet_digit_accuracy = accuracy_score(resnet_test_labels, resnet_test_predictions)
print(f"Frozen ResNet18 test accuracy: {resnet_digit_accuracy:.1%}")


### Compare the results

Record the three accuracies. How many more images did the highest-scoring model classify correctly? Explain what each model receives as input and which layers learn during training.


In [ ]:
print(f"MLP test accuracy: {mlp_test_accuracy:.1%}")
print(f"Simple CNN test accuracy: {simple_digit_accuracy:.1%}")
print(f"ResNet18 test accuracy: {resnet_digit_accuracy:.1%}")
